# Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or python modules that might be required to run this notebook.

# XGBoost Model

In this notebook, you will learn to use XGBoost model and predict sentiment of the news headlines. The steps involved are:
1. Read data
2. Determine target variables
3. Create predictor variables
4. Split data into train and test 
5. Apply Bag of Words on train and test dataset
6. Run XGBoost on the train dataset
7. Predict sentiment scores on the test dataset
8. Analyse the results

## Read data
We read the news sentiment data from file name: "news_headline_sentiments". 

#### Here, we are training the model using only 5000 rows. In case, you want to run the model on the whole dataset; you can download the data from the downloadable zip file in the last section of this course. In code, you need to remove nrows argument.

In [1]:
# Import pandas
import pandas as pd

# Import warning
import warnings
warnings.filterwarnings('ignore')

# Read news sentiment data
news_sentiment_data = pd.read_csv('../data_modules/news_headline_sentiments.csv', nrows=5000)
news_sentiment_data.head()

,news_headline,time_stamp,URL,source_id,sentiment_class,sentiment_scores
0,IMF chief: Beware of global deflation,2014-01-27 10:17:47-08:00,/video/data/2.0/video/business/2014/01/24/intv...,1248,0,0.0000
1,Cars for Sale - ToDrive.com,2014-01-27 10:17:47-08:00,http://www.todrive.com/,1303,0,0.0000
2,Info Edge CFO Raghuvanshi resigns,2014-01-27 10:17:47-08:00,http://www.thehindu.com/business/Industry/info...,1239,-1,-0.3182
3,Weak company earnings drag US stocks mostly lower,2014-01-27 10:17:47-08:00,//www.suntimes.com/business/24993766-420/weak-...,1303,-1,-0.7184
4,Some workers set to reach $100K pay in Boeing ...,2014-01-27 10:17:47-08:00,//www.suntimes.com/business/24725356-420/some-...,1303,-1,-0.0772


## Determine target variables

The target variable, also known as the dependent variable is the variable whose values are to be predicted by predictor variables. Here target variable is to predict sentiment class of the news headlines.  

In [2]:
# Store sentiment_class in y
y = news_sentiment_data.sentiment_class
y=y.map({-1:0,0:1,1:2})

## Create predictor variables

Predictor variable, also known as an independent variable is used to determine the value of the target variable. Here predictor variables are news headlines which are used to predict target variable which is sentiment class of the news headlines.

In [3]:
# Store news headlines in X
X = news_sentiment_data.news_headline

# Convert X in string if the value of x is not string
X = [str(x) if type(x) != str else x for x in X]

## Split data into train and test  

We split the dataset into training and testing dataset. We use 80% of our data to train and the rest 20% to test. You can change the train and test ratio as per choice, but it is advisable to give at least 60% data as train data for good results.

In [4]:
test_ratio = 0.2
train_ratio = 1.0 - test_ratio

num_train = int(train_ratio * len(X))

# X_train and Y_train are training dataset. X_test and Y_test are testing dataset.
X_train = X[:num_train]
y_train = y[:num_train]
X_test = X[num_train:]
y_test = y[num_train:]

## Apply Bag of Words on train and test dataset

We convert training and testing dataset into the bag of words vector. 

For that first we process the news headlines which consists of 
1. converting text to lower case
2. removing stop words such as the, and, of, etc.

We then pass the processed news headlines into `CountVectorizer` method. 


In [5]:
# Import CountVectorizer from sklearn
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer and required arguments to process the data
count_vectorizer = CountVectorizer(stop_words='english', lowercase=True)

# Fit and transform the model on train dataet
X_new_train = count_vectorizer.fit_transform(X_train)

# Transform the test dataset
X_new_test = count_vectorizer.transform(X_test)

## Run XGBoost on the train dataset

To train our model, we pass the the vectors to XGBoost model. We use `XGBClassifier` and pass the following parameters:

1. <b>max_depth:</b>  limits the number of nodes in the tree
2. <b>n_estimators:</b> number of trees to create 

In [6]:
# Import XGBClassifier from xgboost
from xgboost import XGBClassifier

# Instantiate XGBClassifier
xg = XGBClassifier(max_depth = 6, n_estimators = 100, eval_metric ='mlogloss')

# Fit the model on train dataset
xg_model = xg.fit(X_new_train,y_train)

## Predict

Now the model is trained, we predict the sentiment class using the test dataset. We use `predict` function on the test dataset.

In [7]:
# Predict sentiment class on the test dataset
prediction = xg_model.predict(X_new_test)

## Results Analysis
 We use confusion matrix, classification report and prediction accuracy to examine the performance of the classification model.

### Confusion matrix

The confusion matrix is used to describe the performance of classification model on a set of test dataset for which the true values are known. We use `confusion_matrix` function to calculate confusion matrix.

In [8]:
from sklearn.metrics import confusion_matrix 
print(confusion_matrix(y_test, prediction))

[[116 109  17]
 [  8 427   8]
 [  8 125 182]]


### Classification report

In [9]:
from sklearn.metrics import classification_report 
print(classification_report(y_test, prediction))

              precision    recall  f1-score   support

           0       0.88      0.48      0.62       242
           1       0.65      0.96      0.77       443
           2       0.88      0.58      0.70       315

    accuracy                           0.72      1000
   macro avg       0.80      0.67      0.70      1000
weighted avg       0.78      0.72      0.71      1000



### Prediction accuracy

In [10]:
from sklearn.metrics import accuracy_score 
print(accuracy_score(y_test,prediction))

0.725


The XGBoost model trained on the Bag of Words input gives the accuracy of 65%. In the next sections, you will about another word embedding method:TF-IDF. <br><br>